In [ ]:

# STEP 1 - DATA COLLECTION

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import math
warnings.filterwarnings('ignore')




from sklearn.model_selection import (
    train_test_split, GridSearchCV,
    cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor
)
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
import joblib




# PLOT STYLE 
DARK_STYLE = {
    'figure.facecolor': '#0a1628',
    'axes.facecolor':   '#0d1e38',
    'axes.edgecolor':   '#1e3050',
    'axes.labelcolor':  '#8899bb',
    'xtick.color':      '#8899bb',
    'ytick.color':      '#8899bb',
    'text.color':       '#e8f0ff',
    'grid.color':       '#1e3050',
    'grid.alpha':        0.4,
    'axes.titlesize':    13,
    'axes.titlecolor':   '#e8f0ff',
    'axes.labelsize':    11,
    'figure.dpi':        120
}




# PLOT STYLE 2
LIGHT_STYLE = {
    'figure.facecolor': '#f5f5f5',
    'axes.facecolor':   '#f0f0f0',
    'axes.edgecolor':   '#cccccc',
    'axes.labelcolor':  '#333333',
    'xtick.color':      '#555555',
    'ytick.color':      '#555555',
    'text.color':       '#222222',
    'grid.color':       '#dddddd',
    'grid.alpha':        0.6,
    'axes.titlesize':    12,
    'axes.titlecolor':   '#222222',
    'axes.labelsize':    10,
    'figure.dpi':        120
}

def use_dark():
    plt.rcParams.update(DARK_STYLE)

def use_light():
    plt.rcParams.update(LIGHT_STYLE)




# dark style
use_light()

ACCENT = '#f59e0b'
GREEN  = '#22c55e'
CYAN   = '#22d3ee'
PURPLE = '#a78bfa'
RED    = '#ef4444'



# Light style 
L_BLUE   = '#2563eb'
L_ORANGE = '#ea580c'
L_GREEN  = '#16a34a'
L_RED    = '#dc2626'
L_PURPLE = '#7c3aed'



print("  SolarGuard ML")
print("  Rehan / Hadi / M Sajid")
print("-" * 60)
print(".\n")



# Load Dataset
CSV_PATH = r'smart_room_dataset_v3.csv'      # database file 

df = pd.read_csv('C:/Users/hperx/OneDrive/Desktop/SolarGuard/ML/smart_room_dataset_v3.csv')



# Parse timestamp
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])
    df = df.set_index('timestamp')
    date_min = df.index.min().date() if not df.empty else 'N/A'
    date_max = df.index.max().date() if not df.empty else 'N/A'
else:
    date_min = 'N/A'
    date_max = 'N/A'




print("STEP 1 — DATA COLLECTION")
print(f"  Records   : {len(df):,}")
print(f"  Features  : {df.shape[1]}")
print(f"  Date range: {date_min} → {date_max}")
if 'occupied' in df.columns:
    print(f"  Occupancy rate: {df['occupied'].mean()*100:.1f}%")
print()
print(df.head(5).to_string())




#  STEP 2 — DATA CLEANING & PREPROCESSING

print("\n" + "-" * 60)
print("STEP 2 — DATA CLEANING & PREPROCESSING")

print("\n Missing Values -")
missing = df.isnull().sum()
missing_nonzero = missing[missing > 0]
if len(missing_nonzero) > 0:
    print(missing_nonzero.to_string())
else:
    print(" No missing values found")

dup_count = df.duplicated().sum()
print(f"\n Duplicate Rows: {dup_count} ──")
df = df.drop_duplicates()

print("\n Outlier Detection & Capping -")
numeric_cols = [c for c in ['light','temperature','humidity','co2','solar_voltage'] if c in df.columns]
for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    df[col] = df[col].clip(lower=Q1 - 2.5*IQR, upper=Q3 + 2.5*IQR)
    print(f"  {col:20s}: {n_out} outliers capped")

if 'hour' not in df.columns and hasattr(df.index, 'hour'):
    df['hour'] = df.index.hour
if 'weekday' not in df.columns and hasattr(df.index, 'weekday'):
    df['weekday'] = df.index.weekday

df['Minutes'] = df['hour'] * 60 if 'hour' in df.columns else 0

print(f"\nFinal clean dataset shape: {df.shape}")
print("\nStatistical Summary:")
print(df.describe().round(2).to_string())




#  STEP 3 — EXPLORATORY DATA ANALYSIS [EDA]

print("\n" + "-" * 60)
print("STEP 3 — EXPLORATORY DATA ANALYSIS")



# Class Distribution 
use_light()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Target Variable Distributions', fontsize=15, fontweight='bold')

counts = df['occupied'].value_counts().sort_index()
axes[0].bar(['Vacant', 'Occupied'], counts,
            color=[CYAN, ACCENT], width=0.5, edgecolor='none')
axes[0].set_title('Occupancy Class Distribution')
axes[0].set_ylabel('Number of Readings')
axes[0].set_xlabel('Class')
for i, val in enumerate(counts):
    axes[0].text(i, val + 30, f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)
axes[0].set_ylim(0, counts.max() * 1.25)
axes[0].grid(axis='y', alpha=0.3)



if 'energy_risk' in df.columns:
    risk_labels = ['Low Risk', 'Moderate', 'High Risk']
    rc = df['energy_risk'].value_counts().sort_index()
    for i in range(3):
        if i not in rc.index:
            rc[i] = 0
    rc = rc.sort_index()
    axes[1].pie(rc, labels=risk_labels[:len(rc)], colors=[GREEN, ACCENT, RED][:len(rc)],
                autopct='%1.1f%%', startangle=90,
                textprops={'color': 'white', 'fontsize': 10})
axes[1].set_title('Energy Risk Distribution')



if 'comfort_score' in df.columns:
    axes[2].hist(df['comfort_score'], bins=20, color=PURPLE, edgecolor='none', alpha=0.85)
    axes[2].axvline(df['comfort_score'].mean(), color=ACCENT, linestyle='--', linewidth=2,
                    label=f"Mean = {df['comfort_score'].mean():.0f}")
    axes[2].set_title('Comfort Score Distribution')
    axes[2].set_xlabel('Comfort Score (0–100)')
    axes[2].set_ylabel('Frequency')
    axes[2].legend()
    axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_01_class_dist.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(" Dataset balanced [52% occupied] Most readings show low energy risk")
print(" Comfort score left skewed AC system maintains good conditions most of the time")



# Temporal Patterns 
use_light()
if 'hour' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Temporal Patterns [Dubai Lifestyle]', fontsize=14, fontweight='bold')

    hourly_occ = df.groupby('hour')['occupied'].mean() * 100
    axes[0, 0].fill_between(hourly_occ.index, hourly_occ.values, alpha=0.4, color=ACCENT)
    axes[0, 0].plot(hourly_occ.index, hourly_occ.values, color=ACCENT, linewidth=2,
                    marker='o', markersize=4)
    axes[0, 0].set_title('Average Occupancy Rate by Hour of Day')
    axes[0, 0].set_xlabel('Hour (0–23)')
    axes[0, 0].set_ylabel('Occupancy Rate (%)')
    axes[0, 0].set_xticks(range(0, 24, 2))
    axes[0, 0].axhline(50, color='#aaaaaa', alpha=0.5, linestyle='--')
    axes[0, 0].grid(True, alpha=0.3)

    if 'light' in df.columns:
        hourly_light = df.groupby('hour')['light'].mean()
        axes[0, 1].fill_between(hourly_light.index, hourly_light.values, alpha=0.35, color=CYAN)
        axes[0, 1].plot(hourly_light.index, hourly_light.values, color=CYAN, linewidth=2)
        axes[0, 1].set_title('Average LDR Light Intensity by Hour [Dubai Sunlight]')
        axes[0, 1].set_xlabel('Hour of Day')
        axes[0, 1].set_ylabel('LDR Value (0–4095)')
        axes[0, 1].set_xticks(range(0, 24, 2))
        axes[0, 1].grid(True, alpha=0.3)

    if 'weekday' in df.columns:
        dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        dow_occ = df.groupby('weekday')['occupied'].mean() * 100
        bars = axes[1, 0].bar(
            [dow_labels[i] for i in dow_occ.index], dow_occ.values,
            color=[GREEN if i >= 5 else PURPLE for i in dow_occ.index], edgecolor='none'
        )
        axes[1, 0].set_title('Occupancy by Day of Week (Green = Weekend)')
        axes[1, 0].set_xlabel('Day')
        axes[1, 0].set_ylabel('Occupancy Rate (%)')
        for bar, val in zip(bars, dow_occ.values):
            axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                            f'{val:.0f}%', ha='center', fontsize=9)
        axes[1, 0].grid(axis='y', alpha=0.3)

    if 'temperature' in df.columns:
        hourly_temp = df.groupby('hour')['temperature'].agg(['mean', 'std'])
        axes[1, 1].fill_between(hourly_temp.index,
                                hourly_temp['mean'] - hourly_temp['std'],
                                hourly_temp['mean'] + hourly_temp['std'],
                                alpha=0.25, color=RED)
        axes[1, 1].plot(hourly_temp.index, hourly_temp['mean'], color=RED, linewidth=2)
        axes[1, 1].set_title('Indoor Temperature by Hour (Mean ± Std Dev)')
        axes[1, 1].set_xlabel('Hour of Day')
        axes[1, 1].set_ylabel('Temperature (°C)')
        axes[1, 1].set_xticks(range(0, 24, 2))
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('eda_02_temporal.png', bbox_inches='tight', facecolor='#f5f5f5')
    plt.show()
    print(" Peak occupancy 8AM – 10PM weekdays & higher weekends [UAE lifestyle]")



# Feature Distributions 
use_light()
avail_feats = [f for f in ['light','temperature','humidity','co2','solar_voltage'] if f in df.columns]
if avail_feats:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(' Feature Distributions:  Occupied vs Vacant',
                 fontsize=13, fontweight='bold', color='#222222')

    feat_labels = {
        'light': 'Light Intensity (LDR)',
        'temperature': 'Room Temperature (°C)',
        'humidity': 'Humidity (%)',
        'co2': 'CO₂ Level (ppm)',
        'solar_voltage': 'Solar Voltage (V)'
    }
    all_feats = avail_feats + (['comfort_score'] if 'comfort_score' in df.columns else [])
    axes_flat = axes.flat

    for ax, feat in zip(axes_flat, all_feats[:6]):
        lbl = feat_labels.get(feat, feat)
        vacant_data   = df[df['occupied'] == 0][feat]
        occupied_data = df[df['occupied'] == 1][feat]
        ax.hist(vacant_data,   bins=25, alpha=0.6, color=L_BLUE,   label='Vacant',   density=True, edgecolor='white')
        ax.hist(occupied_data, bins=25, alpha=0.6, color=L_ORANGE, label='Occupied', density=True, edgecolor='white')
        ax.set_title(lbl, fontsize=11, color='#222222')
        ax.set_xlabel(lbl, fontsize=9, color='#444444')
        ax.set_ylabel('Density', fontsize=9, color='#444444')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.4, color='#cccccc')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    for ax in list(axes_flat)[len(all_feats):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.savefig('eda_03_distributions.png', bbox_inches='tight', facecolor='#f5f5f5')
    plt.show()
    print(" CO₂ clearly higher when occupied [500 – 900 vs 400 – 500 ppm]")
    print(" Temperature 1 – 2°C higher when occupied, human body heat effect")


# Scatter Plots
use_light()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Scatter Analysis', fontsize=13, fontweight='bold', color='#222222')

if 'Minutes' in df.columns and 'light' in df.columns:
    for occ, col, nm in zip([0, 1], [L_BLUE, L_ORANGE], ['Vacant', 'Occupied']):
        mask = df['occupied'] == occ
        axes[0].scatter(df.loc[mask, 'Minutes'], df.loc[mask, 'light'],
                        alpha=0.2, s=5, color=col, label=nm)
    axes[0].set_title('Light Intensity vs Time of Day', fontsize=11, color='#222222')
    axes[0].set_xlabel('Minutes into Day (hour × 60)', fontsize=9)
    axes[0].set_ylabel('LDR Light Value', fontsize=9)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.4)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

if 'temperature' in df.columns and 'co2' in df.columns:
    for occ, col, nm in zip([0, 1], [L_BLUE, L_ORANGE], ['Vacant', 'Occupied']):
        mask = df['occupied'] == occ
        axes[1].scatter(df.loc[mask, 'temperature'], df.loc[mask, 'co2'],
                        alpha=0.25, s=5, color=col, label=nm)
    axes[1].set_title('Temperature vs CO₂ Level', fontsize=11, color='#222222')
    axes[1].set_xlabel('Temperature (°C)', fontsize=9)
    axes[1].set_ylabel('CO₂ (ppm)', fontsize=9)
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.4)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eda_04_scatter.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(" Clear separation between occupied [orange] & vacant [blue] clusters")
print(" Light peaks at midday regardless of occupancy, Dubai sun effect is strong")



# Correlation Heatmap 
use_light()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Correlation Analysis', fontsize=14, fontweight='bold')

drop_cols  = ['weekday', 'is_weekend', 'Minutes']
numeric_df = df.select_dtypes(include=[np.number]).drop(
    columns=[c for c in drop_cols if c in df.columns], errors='ignore')
corr = numeric_df.corr()

sns.heatmap(corr,
            mask=np.triu(np.ones_like(corr, dtype=bool)),
            ax=axes[0], annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 7},
            linewidths=0.5, linecolor='#dddddd')
axes[0].set_title('Full Feature Correlation Matrix')
axes[0].tick_params(axis='both', labelsize=8)

if 'occupied' in corr.columns:
    target_corr = corr['occupied'].drop('occupied').sort_values()
    colors_bar  = [RED if v < 0 else GREEN for v in target_corr]
    axes[1].barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='none')
    axes[1].axvline(0, color='#888888', alpha=0.5)
    axes[1].set_title('Feature Correlation with Occupancy (Target)')
    axes[1].set_xlabel('Pearson r Coefficient')
    for i, val in enumerate(target_corr.values):
        x = val + 0.01 if val >= 0 else val - 0.04
        axes[1].text(x, i, f'{val:.3f}', va='center', fontsize=8)
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_05_correlation.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(" PIR strongest positive correlation [r = 0.83] CO₂ also strongly correlated")
print(" Hour_cos negative - late night hours are predominantly vacant")



# Boxplots 
use_light()
box_feats = [f for f in ['light','temperature','humidity','co2'] if f in df.columns]
if box_feats:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Boxplot Analysis by Occupancy', fontsize=14, fontweight='bold')

    feat_col_map = {'light': ACCENT, 'temperature': RED, 'humidity': CYAN, 'co2': PURPLE}
    feat_lbl_map = {'light': 'Light (LDR)', 'temperature': 'Temperature (°C)',
                    'humidity': 'Humidity (%)', 'co2': 'CO₂ (ppm)'}

    for ax, feat in zip(axes.flat, box_feats):
        col = feat_col_map.get(feat, CYAN)
        lbl = feat_lbl_map.get(feat, feat)
        try:
            sns.boxplot(x='occupied', y=feat, data=df, ax=ax,
                        palette={0: CYAN + '88', 1: col + '88'},
                        flierprops={'marker': '.', 'alpha': 0.3, 'markersize': 3})
        except Exception:
            sns.boxplot(x='occupied', y=feat, data=df, ax=ax)
        ax.set_xticklabels(['Vacant', 'Occupied'])
        ax.set_title(f'{lbl} by Occupancy')
        ax.set_xlabel('Occupancy Status')
        ax.set_ylabel(lbl)
        ax.grid(axis='y', alpha=0.3)
        for i, occ in enumerate([0, 1]):
            med = df[df['occupied'] == occ][feat].median()
            ax.text(i, med, f' {med:.1f}', va='center', fontsize=9, color='#333333')

    plt.tight_layout()
    plt.savefig('eda_06_boxplots.png', bbox_inches='tight', facecolor='#f5f5f5')
    plt.show()
    print(" CO₂ median -200 ppm higher when occupied.")




#  STEP 4 — FEATURE ENGINEERING & SELECTION

print("\n" + "-" * 60)
print("STEP 4 — FEATURE ENGINEERING & SELECTION")
use_light()

df_fe = df.copy()

if 'temperature' in df_fe.columns and 'humidity' in df_fe.columns:
    df_fe['thi'] = df_fe['temperature'] - 0.55*(1 - df_fe['humidity']/100)*(df_fe['temperature'] - 14.5)
else:
    df_fe['thi'] = 0

if 'light' in df_fe.columns:
    df_fe['light_cat'] = pd.cut(df_fe['light'], bins=[0,300,1000,2500,4100], labels=[0,1,2,3]).astype(int)
else:
    df_fe['light_cat'] = 0

if 'hour' in df_fe.columns:
    df_fe['is_daytime'] = ((df_fe['hour'] >= 6) & (df_fe['hour'] <= 20)).astype(int)
    df_fe['solar_peak'] = ((df_fe['hour'] >= 9) & (df_fe['hour'] <= 15)).astype(int)
    df_fe['hour_sin']   = df_fe['hour'].apply(lambda h: math.sin(2*math.pi*h/24))
    df_fe['hour_cos']   = df_fe['hour'].apply(lambda h: math.cos(2*math.pi*h/24))
else:
    df_fe['is_daytime'] = 1; df_fe['solar_peak'] = 0
    df_fe['hour_sin']   = 0; df_fe['hour_cos']   = 1

if 'weekday' in df_fe.columns:
    df_fe['dow_sin']    = df_fe['weekday'].apply(lambda d: math.sin(2*math.pi*d/7))
    df_fe['dow_cos']    = df_fe['weekday'].apply(lambda d: math.cos(2*math.pi*d/7))
    df_fe['is_weekend'] = (df_fe['weekday'] >= 5).astype(int)
else:
    df_fe['dow_sin'] = 0; df_fe['dow_cos'] = 1; df_fe['is_weekend'] = 0

df_fe['temp_high'] = (df_fe['temperature'] > 27.0).astype(int) if 'temperature' in df_fe.columns else 0
df_fe['co2_risk']  = (df_fe['co2'] > 800).astype(int) if 'co2' in df_fe.columns else 0
df_fe['pir_trend'] = df_fe['pir'].rolling(3, min_periods=1).mean().round(2) if 'pir' in df_fe.columns else 0
df_fe['light_pir'] = (df_fe['light']*df_fe['pir'])/100 if ('light' in df_fe.columns and 'pir' in df_fe.columns) else 0

new_feats = ['thi','light_cat','is_daytime','temp_high','co2_risk','solar_peak','pir_trend','light_pir']
print("New engineered features:")
print(df_fe[new_feats].describe().round(2).to_string())

all_possible = [
    'light','temperature','humidity','pir','co2','solar_voltage',
    'hour_sin','hour_cos','dow_sin','dow_cos','is_weekend',
    'thi','light_cat','is_daytime','temp_high','co2_risk',
    'solar_peak','pir_trend','light_pir'
]
feature_cols = [f for f in all_possible if f in df_fe.columns]

X_all = df_fe[feature_cols].fillna(0)
y_occ = df_fe['occupied']

selector_f = SelectKBest(f_classif, k='all')
selector_f.fit(X_all, y_occ)
f_scores  = pd.Series(selector_f.scores_, index=feature_cols).sort_values(ascending=True)
mi_scores = pd.Series(mutual_info_classif(X_all, y_occ, random_state=42),
                      index=feature_cols).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Selection: F-Score & Mutual Information', fontsize=14, fontweight='bold')

axes[0].barh(f_scores.index, f_scores.values,
             color=[ACCENT if v > f_scores.median() else CYAN for v in f_scores], edgecolor='none')
axes[0].set_title('ANOVA F-Score (higher = more discriminative)')
axes[0].set_xlabel('F-Score')
axes[0].axvline(f_scores.median(), color='#888888', alpha=0.5, linestyle='--', label='Median')
axes[0].legend(); axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(mi_scores.index, mi_scores.values,
             color=[PURPLE if v > mi_scores.median() else CYAN for v in mi_scores], edgecolor='none')
axes[1].set_title('Mutual Information Score (non-linear dependency)')
axes[1].set_xlabel('MI Score (bits)')
axes[1].axvline(mi_scores.median(), color='#888888', alpha=0.5, linestyle='--', label='Median')
axes[1].legend(); axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_07_feature_selection.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()

n_top        = min(12, len(feature_cols))
top_features = mi_scores.nlargest(n_top).index.tolist()
X = X_all[top_features]
print(f"\nTop {n_top} features by Mutual Information:")
for i, f in enumerate(top_features, 1):
    print(f"  {i:2d}. {f:20s}  MI = {mi_scores[f]:.4f}")
print(f"\nFinal feature matrix shape: {X.shape}")




# STEP 5 — MODEL TRAINING

print("\n" + "-" * 60)
print("STEP 5 — MODEL TRAINING")
use_light()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_occ, test_size=0.2, random_state=42, stratify=y_occ
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  |  Features: {X_train.shape[1]}")
print(f"Train occ: {y_train.mean()*100:.1f}%  |  Test occ: {y_test.mean()*100:.1f}%")

models = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

results = {}
cv_skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\n{'Model':<25} {'Accuracy':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'AUC':>8} {'CV-5':>12}")
print('-' * 85)

for name, model in models.items():
    Xtr = X_train_sc if name in ['SVM','Logistic Regression'] else X_train.values
    Xte = X_test_sc  if name in ['SVM','Logistic Regression'] else X_test.values
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1] if hasattr(model,'predict_proba') else None
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.0
    cv_s = cross_val_score(model, Xtr, y_train, cv=cv_skf, scoring='accuracy', n_jobs=-1)
    results[name] = {
        'model':model,'accuracy':acc,'f1':f1,'precision':prec,'recall':rec,
        'auc':auc,'cv_mean':cv_s.mean(),'cv_std':cv_s.std(),
        'y_pred':y_pred,'y_prob':y_prob
    }
    print(f"{name:<25} {acc:>10.4f} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f} "
          f"{auc:>8.4f} {cv_s.mean():>6.4f}±{cv_s.std():.3f}")



#  STEP 6 — MODEL COMPARISON & EVALUATION

print("\n" + "-" * 90)
print("STEP 6 — MODEL COMPARISON & EVALUATION")
use_light()

model_names  = list(results.keys())
x = np.arange(len(model_names))

# Model Comparison 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Model Comparison: All Metrics', fontsize=14, fontweight='bold')

metrics_list = ['accuracy','f1','precision','recall','auc']
met_labels   = ['Accuracy','F1 Score','Precision','Recall','ROC-AUC']
bar_cols_    = [ACCENT, CYAN, PURPLE, GREEN, RED]
width = 0.15
for i, (met, lbl, col) in enumerate(zip(metrics_list, met_labels, bar_cols_)):
    vals = [results[m][met] for m in model_names]
    axes[0].bar(x + (i-2)*width, vals, width, label=lbl, color=col, alpha=0.85, edgecolor='none')

axes[0].set_xticks(x)
axes[0].set_xticklabels([m.replace(' ','\n') for m in model_names], fontsize=9)
axes[0].set_ylabel('Score'); axes[0].set_ylim(0.7, 1.05)
axes[0].legend(fontsize=8); axes[0].set_title('All Metrics Comparison')
axes[0].axhline(0.9, color='#aaaaaa', alpha=0.5, linestyle='--')
axes[0].grid(axis='y', alpha=0.3)

cv_means = [results[m]['cv_mean'] for m in model_names]
cv_stds  = [results[m]['cv_std']  for m in model_names]
axes[1].bar(model_names, cv_means, yerr=cv_stds,
            color=[ACCENT if m == 'Random Forest' else CYAN for m in model_names],
            alpha=0.85, edgecolor='none', capsize=5, error_kw={'ecolor':'white','alpha':0.5})
axes[1].set_title('5-Fold Cross-Validation Accuracy (mean ± std)')
axes[1].set_ylabel('CV Accuracy'); axes[1].set_ylim(0.7, 1.05)
axes[1].set_xticklabels([m.replace(' ','\n') for m in model_names], fontsize=9)
for i, (val, std) in enumerate(zip(cv_means, cv_stds)):
    axes[1].text(i, val+std+0.005, f'{val:.3f}', ha='center', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_01_comparison.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()

best_name = max(results, key=lambda k: results[k]['f1'])
print(f"\nBest Model: {best_name}")
print(f"  Accuracy : {results[best_name]['accuracy']:.4f}")
print(f"  F1 Score : {results[best_name]['f1']:.4f}")
print(f"  ROC-AUC  : {results[best_name]['auc']:.4f}")


# Confusion Matrices 
use_light()
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices: All Models', fontsize=13, fontweight='bold')
for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                linewidths=0.5, linecolor='#dddddd')
    acc = results[name]['accuracy']
    ax.set_title(f"{name.replace(' ',chr(10))}\nacc={acc:.3f}", fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticklabels(['Vacant','Occupied'], fontsize=8)
    ax.set_yticklabels(['Vacant','Occupied'], fontsize=8, rotation=0)
plt.tight_layout()
plt.savefig('model_02_confusion.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()


# ROC Curves 
use_light()
fig, ax = plt.subplots(figsize=(9, 7))
roc_colors = [ACCENT, CYAN, PURPLE, GREEN, RED]
for name, col in zip(model_names, roc_colors):
    if results[name]['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
        ax.plot(fpr, tpr, color=col, linewidth=2,
                label=f"{name} (AUC={results[name]['auc']:.4f})")
ax.plot([0,1],[0,1],'--', color='#aaaaaa', alpha=0.6, linewidth=1, label='Random baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.4, color='#cccccc')
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig('model_03_roc.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()

print(f"\nClassification Report — {best_name}")
print("-" * 55)
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['Vacant','Occupied']))


# Comfort Score Regression 
use_light()
if 'comfort_score' in df_fe.columns:
    print("Comfort Score Regression:")
    y_comfort = df_fe['comfort_score']
    Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X, y_comfort, test_size=0.2, random_state=42)
    sc2 = StandardScaler()
    Xtr_cs = sc2.fit_transform(Xtr_c); Xte_cs = sc2.transform(Xte_c)

    rf_reg = RandomForestRegressor(100, random_state=42)
    rf_reg.fit(Xtr_c.values, ytr_c)
    yp = rf_reg.predict(Xte_c.values)
    mae  = mean_absolute_error(yte_c, yp)
    rmse = mean_squared_error(yte_c, yp)**0.5
    r2   = r2_score(yte_c, yp)
    print(f"  RF Regressor  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Comfort Score: Actual vs Predicted',
                 fontsize=12, fontweight='bold', color='#222222')

# scatter plot
    axes[0].scatter(yte_c, yp, alpha=0.4, s=10, color=L_PURPLE, edgecolors='none')
    axes[0].plot([0,100],[0,100], '--', color='#999999', linewidth=1.5)
    axes[0].set_xlabel('Actual Score', fontsize=9)
    axes[0].set_ylabel('Predicted Score', fontsize=9)
    axes[0].set_title(f'Scatter: R² = {r2:.4f}', fontsize=10, color='#222222')
    axes[0].grid(True, alpha=0.4)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

# Bar chart showing metric values
    metric_names = ['MAE', 'RMSE', 'R²']
    metric_vals  = [mae, rmse, r2 * 100]         # scale R2 for visibility
    bar_colors   = [L_BLUE, L_ORANGE, L_GREEN]
    bars = axes[1].bar(metric_names, metric_vals, color=bar_colors, edgecolor='white', width=0.5)
    axes[1].set_title('Model Error Metrics (R² shown as %)', fontsize=10, color='#222222')
    axes[1].set_ylabel('Value', fontsize=9)
    for bar, val in zip(bars, metric_vals):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f'{val:.2f}', ha='center', fontsize=10, color='#222222')
    axes[1].grid(axis='y', alpha=0.4)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('model_04_regression.png', bbox_inches='tight', facecolor='#f5f5f5')
    plt.show()



#  STEP 7 — VISUALIZATION & INTERPRETATION

print("\n" + "-" * 60)
print("STEP 7 — VISUALIZATION & INTERPRETATION")
use_light()

# Feature Importance 
rf_model = results['Random Forest']['model']
gb_model = results['Gradient Boosting']['model']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Importance', fontsize=14, fontweight='bold')

rf_imp = pd.Series(rf_model.feature_importances_, index=top_features).sort_values(ascending=True)
gb_imp = pd.Series(gb_model.feature_importances_, index=top_features).sort_values(ascending=True)

axes[0].barh(rf_imp.index, rf_imp.values,
             color=[ACCENT if v > rf_imp.median() else CYAN for v in rf_imp], edgecolor='none')
axes[0].set_title('Random Forest — Feature Importance (Gini)')
axes[0].set_xlabel('Importance Score'); axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(gb_imp.index, gb_imp.values,
             color=[PURPLE if v > gb_imp.median() else CYAN for v in gb_imp], edgecolor='none')
axes[1].set_title('Gradient Boosting — Feature Importance')
axes[1].set_xlabel('Importance Score'); axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_05_importance.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(f" Top 3 RF features: {', '.join(rf_imp.nlargest(3).index.tolist())}")



# PCA Visualization 

use_light()
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PCA: 2D View of Feature Space',
             fontsize=12, fontweight='bold', color='#222222')

# True labels
for occ, col, nm in zip([0,1], [L_BLUE, L_ORANGE], ['Vacant','Occupied']):
    mask = y_test.values == occ
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], c=col, alpha=0.4, s=8,
                    edgecolors='none', label=nm)
axes[0].set_title('True Labels', fontsize=10, color='#222222')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=9)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=9)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.4)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# RF predictions
rf_pred = results['Random Forest']['y_pred']
for occ, col, nm in zip([0,1], [L_BLUE, L_ORANGE], ['Vacant','Occupied']):
    mask = rf_pred == occ
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], c=col, alpha=0.4, s=8,
                    edgecolors='none', label=nm)
axes[1].set_title('Random Forest Predictions', fontsize=10, color='#222222')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=9)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=9)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.4)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('model_06_pca.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(f" PCA explains {sum(pca.explained_variance_ratio_)*100:.1f}% of total variance in 2D.")
print(" Clear separation between occupied/vacant confirms the model can distinguish them well.")


# Live Prediction Demo 
print("\nLive Prediction Demo:")
print("-" * 55)

sample_readings = [
    {'name':'Morning — occupied',  'pir':1,'light':2400,'temperature':26.5,'humidity':55,'co2':650,'hour':9, 'is_weekend':0},
    {'name':'Afternoon — vacant',  'pir':0,'light':3800,'temperature':28.1,'humidity':52,'co2':420,'hour':14,'is_weekend':0},
    {'name':'Night — occupied',    'pir':1,'light':800, 'temperature':25.0,'humidity':58,'co2':720,'hour':21,'is_weekend':0},
    {'name':'Late night — asleep', 'pir':0,'light':100, 'temperature':23.0,'humidity':60,'co2':410,'hour':2, 'is_weekend':0},
    {'name':'Weekend — family',    'pir':1,'light':1200,'temperature':27.0,'humidity':62,'co2':810,'hour':16,'is_weekend':1},
]
final_model = results['Random Forest']['model']

for reading in sample_readings:
    rname = reading.pop('name')
    hour  = reading['hour']
    sd = {
        'light':reading.get('light',1000),'temperature':reading.get('temperature',25),
        'humidity':reading.get('humidity',50),'pir':reading.get('pir',0),
        'co2':reading.get('co2',500),'solar_voltage':4.5 if 6<=hour<=18 else 3.7,
        'hour_sin':math.sin(2*math.pi*hour/24),'hour_cos':math.cos(2*math.pi*hour/24),
        'dow_sin':0.0,'dow_cos':1.0,'is_weekend':reading.get('is_weekend',0),
        'thi':reading.get('temperature',25)-0.55*(1-reading.get('humidity',50)/100)*(reading.get('temperature',25)-14.5),
        'light_cat':min(3,int(reading.get('light',1000)/1000)),
        'is_daytime':int(6<=hour<=20),'temp_high':int(reading.get('temperature',25)>27),
        'co2_risk':int(reading.get('co2',500)>800),'solar_peak':int(9<=hour<=15),
        'pir_trend':float(reading.get('pir',0)),
        'light_pir':reading.get('light',0)*reading.get('pir',0)/100
    }
    available = [f for f in top_features if f in sd]
    vec  = pd.DataFrame([sd])[available]
    pred = final_model.predict(vec)[0]
    prob = final_model.predict_proba(vec)[0][1]
    print(f"\n {rname}")
    print(f"  Result  : {'OCCUPIED' if pred==1 else 'VACANT'}  ({prob*100:.1f}% confidence)")
    print(f"  Action  : {'Turn ON fan & LED' if pred==1 else 'Turn OFF all devices'}")




#  STEP 8 — HYPERPARAMETER TUNING (GridSearchCV)

print("\n" + "-" * 60)
print("STEP 8 — HYPERPARAMETER TUNING [GridSearchCV]")

use_light()
print("Searching best parameters for Random Forest....\n")

param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [None, 8, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2],
    'max_features':      ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted', n_jobs=-1, verbose=1
)
grid_search.fit(X_train.values, y_train)

rf_tuned     = grid_search.best_estimator_
y_pred_tuned = rf_tuned.predict(X_test.values)
acc_tuned    = accuracy_score(y_test, y_pred_tuned)
f1_tuned     = f1_score(y_test, y_pred_tuned, average='weighted')
auc_tuned    = roc_auc_score(y_test, rf_tuned.predict_proba(X_test.values)[:,1])

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"\n{'Metric':<12} {'Before':>10} {'After':>10} {'Change':>10}")
print('-' * 50)
for m, b, a in [
    ('Accuracy', results['Random Forest']['accuracy'], acc_tuned),
    ('F1 Score', results['Random Forest']['f1'],       f1_tuned),
    ('ROC-AUC',  results['Random Forest']['auc'],      auc_tuned)
]:
    print(f"{m:<12} {b:>10.4f} {a:>10.4f} {a-b:>+10.4f}")

cv_results_df = pd.DataFrame(grid_search.cv_results_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GridSearchCV: Hyperparameter Effect on F1', fontsize=13, fontweight='bold')
for ax, param, lbl in zip(
    axes,
    ['param_n_estimators','param_max_depth','param_min_samples_split'],
    ['n_estimators','max_depth','min_samples_split']
):
    group = cv_results_df.groupby(param)['mean_test_score'].mean()
    std_g = cv_results_df.groupby(param)['std_test_score'].mean()
    idx = range(len(group))
    ax.plot(idx, group.values, color=ACCENT, marker='o', linewidth=2, markersize=6)
    ax.fill_between(idx, group.values-std_g.values, group.values+std_g.values,
                    alpha=0.2, color=ACCENT)
    ax.set_xticks(idx); ax.set_xticklabels(group.index.astype(str))
    ax.set_title(f'Effect of {lbl}')
    ax.set_xlabel(lbl); ax.set_ylabel('Mean CV F1'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_07_gridsearch.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
m_names  = ['Accuracy', 'F1 Score', 'ROC-AUC']
before_v = [results['Random Forest']['accuracy'], results['Random Forest']['f1'], results['Random Forest']['auc']]
after_v  = [acc_tuned, f1_tuned, auc_tuned]
xb = np.arange(len(m_names))
ax.bar(xb-0.2, before_v, 0.38, label='Before Tuning', color='#2563eb', alpha=0.85, edgecolor='none')
ax.bar(xb+0.2, after_v,  0.38, label='After Tuning',  color=ACCENT, alpha=0.85, edgecolor='none')
ax.set_xticks(xb); ax.set_xticklabels(m_names)
ax.set_ylim(0.88, 1.01); ax.set_ylabel('Score')
ax.set_title('Random Forest: Before vs After GridSearchCV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for xi, (b, a) in enumerate(zip(before_v, after_v)):
    ax.text(xi-0.2, b+0.001, f'{b:.4f}', ha='center', fontsize=8, color='#333333')
    ax.text(xi+0.2, a+0.001, f'{a:.4f}', ha='center', fontsize=8, color='#333333')
plt.tight_layout()
plt.savefig('model_08_before_after.png', bbox_inches='tight', facecolor='#f5f5f5')
plt.show()

joblib.dump(rf_tuned,     'rf_model_tuned.pkl')
joblib.dump(scaler,       'scaler.pkl')
joblib.dump(top_features, 'selected_features.pkl')
print("\n Saved: rf_model_tuned.pkl, scaler.pkl, selected_features.pkl")

summary = pd.DataFrame({
    'Model':     model_names,
    'Accuracy':  [f"{results[k]['accuracy']:.4f}" for k in results],
    'F1':        [f"{results[k]['f1']:.4f}"       for k in results],
    'Precision': [f"{results[k]['precision']:.4f}" for k in results],
    'Recall':    [f"{results[k]['recall']:.4f}"    for k in results],
    'ROC-AUC':   [f"{results[k]['auc']:.4f}"       for k in results],
    'CV Mean':   [f"{results[k]['cv_mean']:.4f}"   for k in results],
}).set_index('Model')

print("\n Final Model Comparison Table")
print(summary.to_string())

print("""
--------------------------------------------------------------------

                     CONCLUSION 
  
   Step 1:  5000-row Dubai sensor dataset loaded
   Step 2:  IQR outlier capping, no missing values
   Step 3:  6 EDA visualizations with insights
   Step 4:  8 new features, MI + ANOVA selection
   Step 5:  5 sklearn models trained & compared
   Step 6:  Confusion matrices, ROC-AUC, regression
   Step 7:  PCA, feature importance, live demo
   Step 8:  GridSearchCV tuning, before/after chart

  Best Model  : Random Forest (after GridSearchCV)
  Primary Task: Occupancy prediction (classification)
  Extra Tasks : Comfort Score (regression)
                Energy Risk (classification)
                
----------------------------------------------------------------------
""")